<a href="https://colab.research.google.com/github/pareshmishra23/4k-genration/blob/main/4k_Video_Upscaler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 4K Video Upscaler (Real-ESRGAN)

Upscale your videos up to 4K using Real-ESRGAN. Works automatically with GPU (T4/V100) or CPU.

**Before running:** Go to **Runtime → Change runtime type → GPU (T4)** for best performance.

Repository: [github.com/pareshmishra23/4k-genration](https://github.com/pareshmishra23/4k-genration)


In [ ]:
# Cell 1: Setup — Install all dependencies and patch compatibility issues
import os, sys, pathlib

# Clone repositories
!git clone https://github.com/xinntao/Real-ESRGAN.git

# Install PyTorch + dependencies
device = 'cuda'
try:
    import torch
    if torch.cuda.is_available():
        device = 'cuda'
        print('✅ GPU detected:', torch.cuda.get_device_name(0))
    else:
        device = 'cpu'
        print('⚠️ No GPU detected, using CPU')
except:
    device = 'cpu'
    print('⚠️ No GPU detected, using CPU')

if device == 'cuda':
    !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
else:
    !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cpu

!pip install -q basicsr facexlib gfpgan ffmpeg ffmpeg-python tqdm
!pip install -q -r Real-ESRGAN/requirements.txt
!pip install -q -e Real-ESRGAN
!pip install "numpy<2"

# Patch deprecated torchvision import in basicsr
import glob
basicsr_path = glob.glob('/usr/local/lib/python*/dist-packages/basicsr/data/degradations.py')
if basicsr_path:
    degradations = pathlib.Path(basicsr_path[0])
    content = degradations.read_text()
    if 'functional_tensor' in content:
        content = content.replace(
            'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
            'from torchvision.transforms.functional import rgb_to_grayscale'
        )
        degradations.write_text(content)
        print('✅ Patched basicsr compatibility')

# Verify installation
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan.archs.srvgg_arch import SRVGGNetCompact
print(f'✅ All dependencies installed successfully!')
print(f'✅ Device: {device}')


# ⚙️ Configuration

Set your video path, output directory, resolution, and model using the form below.

In [ ]:
# @title ⚙️ Configure Settings
video_path = "/content/gdrive/MyDrive/content/video.mp4" #@param {type:"string"}
output_dir = "/content/gdrive/MyDrive/content/" #@param {type:"string"}
resolution = "4k (3840 x 2160)" #@param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)", "2 x original", "3 x original", "4 x original"] {type:"string"}
model = "RealESRGAN_x4plus_anime_6B" #@param ["RealESRGAN_x4plus", "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3", "RealESRNet_x4plus", "RealESRGAN_x2plus", "realesr-general-x4v3"]
tile = 128 #@param {type:"integer"}

# Validation
if not os.path.exists(video_path):
    print(f'❌ Video file not found: {video_path}')
    print('Please upload the video to your Google Drive or change the path above.')
else:
    print(f'✅ Video found: {video_path}')
    import cv2
    cap = cv2.VideoCapture(video_path)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frames / fps
    cap.release()
    print(f'✅ Resolution: {w}x{h} | FPS: {fps} | Duration: {duration:.1f}s | Frames: {frames}')
    print(f'✅ Output will be saved to: {output_dir}')
    print(f'✅ Target: {resolution} | Model: {model} | Tile: {tile}')


In [ ]:
# Cell 3: Run Upscaling
import sys
sys.path.insert(0, '/content/Real-ESRGAN')

import torch
import cv2
import numpy as np
import subprocess
from basicsr.utils.download_util import load_file_from_url
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer
from realesrgan.archs.srvgg_arch import SRVGGNetCompact
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Get video info
cap = cv2.VideoCapture(video_path)
video_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

# Calculate output resolution
resolution_map = {
    "FHD (1920 x 1080)": (1920, 1080),
    "2k (2560 x 1440)": (2560, 1440),
    "4k (3840 x 2160)": (3840, 2160),
}

aspect_ratio = float(video_width / video_height)

if resolution in resolution_map:
    final_width, final_height = resolution_map[resolution]
elif "x original" in resolution:
    multiplier = int(resolution.split("x")[0].strip())
    final_width = multiplier * video_width
    final_height = multiplier * video_height

if aspect_ratio == 1.0 and "original" not in resolution:
    final_height = final_width
if aspect_ratio < 1.0 and "original" not in resolution:
    final_width, final_height = final_height, final_width

scale_factor = max(final_width / video_width, final_height / video_height)

while True:
    sw = int(video_width * scale_factor)
    sh = int(video_height * scale_factor)
    if sw % 2 == 0 and sh % 2 == 0:
        break
    scale_factor += 0.01

print(f'Input:    {video_width}x{video_height}')
print(f'Target:   {final_width}x{final_height}')
print(f'Scale:    {scale_factor:.2f}x')
print(f'Model:    {model}')
print(f'Device:   {device}')
print(f'Tile:     {tile}')
print(f'Frames:   {total_frames}')
print(f'FPS:      {fps}')
print('—' * 40)

# Setup model
model_configs = {
    "RealESRGAN_x4plus": (RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4), 4,
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"),
    "RealESRNet_x4plus": (RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4), 4,
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.1/RealESRNet_x4plus.pth"),
    "RealESRGAN_x4plus_anime_6B": (RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=6, num_grow_ch=32, scale=4), 4,
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth"),
    "RealESRGAN_x2plus": (RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=2), 2,
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth"),
    "realesr-animevideov3": (SRVGGNetCompact(num_in_ch=3, num_out_ch=3, num_feat=64, num_conv=16, upscale=4, act_type='prelu'), 4,
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth"),
}

net, netscale, file_url = model_configs[model]
model_path = load_file_from_url(url=file_url, model_dir='/content/Real-ESRGAN/weights', progress=True, file_name=None)

upsampler = RealESRGANer(
    scale=netscale, model_path=model_path, dni_weight=None, model=net,
    tile=tile, tile_pad=10, pre_pad=0,
    half=(device == 'cuda'), device=torch.device(device)
)

# Process video
os.makedirs(output_dir, exist_ok=True)
video_name = os.path.splitext(os.path.basename(video_path))[0]
out_width = int(video_width * scale_factor)
out_height = int(video_height * scale_factor)

cap = cv2.VideoCapture(video_path)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
temp_path = os.path.join(output_dir, f'{video_name}_temp.mp4')
out_writer = cv2.VideoWriter(temp_path, fourcc, fps, (out_width, out_height))

pbar = tqdm(total=total_frames, desc='Upscaling')
frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    try:
        output, _ = upsampler.enhance(frame, outscale=scale_factor)
        out_writer.write(output)
    except RuntimeError as e:
        print(f'Error on frame {frame_idx}: {e}')
        print('Try reducing tile size.')
    pbar.update(1)
    frame_idx += 1

cap.release()
out_writer.release()
pbar.close()

# Crop to final resolution
final_path = os.path.join(output_dir, f'{video_name}_upscaled_{final_width}_{final_height}.mp4')
if "original" not in resolution:
    print('Cropping to final resolution...')
    cmd = ['ffmpeg', '-y', '-i', temp_path,
           '-filter:v', f'crop={final_width}:{final_height}:(in_w-{final_width})/2:(in_h-{final_height})/2',
           '-c:v', 'libx264', '-pix_fmt', 'yuv420p', final_path]
    subprocess.run(cmd, check=True)
else:
    os.replace(temp_path, final_path)

if os.path.exists(temp_path):
    os.remove(temp_path)

print(f'✅ Done! Output: {final_path}')
print(f'✅ Final resolution: {final_width}x{final_height}')


In [ ]:
# Cell 4: Download the upscaled video
import glob
from google.colab import files

download_to_pc = True #@param {type:"boolean"}

output_files = glob.glob(os.path.join(output_dir, '*upscaled*.mp4'))
if output_files:
    for f in output_files:
        print(f'✅ Output found: {f}')
        if download_to_pc:
            files.download(f)
        else:
            print('Skipping download (uncheck the box above to download)')
else:
    print('❌ No output files found. Did Cell 3 complete successfully?')

# Show file size
for f in output_files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f'📦 File size: {size_mb:.1f} MB')
